# 02 · NEU Steel Defects - Preprocessing & DataLoaders

The `train/images/[class]/` layout is compatible with torchvision `ImageFolder`.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
device = get_device()
ASSETS = "cv/neu"


In [ ]:
DATA_DIR  = Path('../../../data/04_neu/raw')
TRAIN_DIR = DATA_DIR / 'train'  / 'images'   # ImageFolder-compatible
VAL_DIR   = DATA_DIR / 'validation' / 'images'

# Hyperparameters
IMG_SIZE    = 224  # NEU images ~200px, resize to 224
BATCH_SIZE  = 32
NUM_WORKERS = recommended_num_workers(device)
PIN_MEMORY  = device.type == "cuda"
SEED        = 42
torch.manual_seed(SEED)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # NEU images are grayscale
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print('Transforms defined.')

In [ ]:
train_set = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_set   = datasets.ImageFolder(root=VAL_DIR,   transform=val_transforms)

CLASSES    = train_set.classes
NUM_CLASSES = len(CLASSES)
print(f'Classes ({NUM_CLASSES}): {CLASSES}')
print(f'Class to idx: {train_set.class_to_idx}')
print(f'Train: {len(train_set)} | Val: {len(val_set)}')

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

images, labels = next(iter(train_loader))
print(f'Batch shape: {images.shape}')  # [32, 3, 224, 224]
print(f'Labels: {[CLASSES[l] for l in labels[:6].tolist()]}')

## Summary

| Parameter | Value |
|---|---|
| Input | Grayscale → 3-channel (for the ImageNet backbone) |
| Resize | 200×200 → 224×224 |

➡️ **Next step:** `03_neu_modeling.ipynb`